# Part 2 — MOFA tools over MCP (FastMCP + langchain-mcp-adapters)

Same analysis tools as [`01_tools_v1`](01_tools_v1.ipynb), but now served by a
**FastMCP** server (`server/mofa_mcp_server.py`) and pulled into the agent via
**langchain-mcp-adapters**. The agent code is almost identical to notebook 01;
only the *source* of the tools changes.

```text
LLM client  ->  MCP server (FastMCP)  ->  cached MOFA model  ->  structured result  ->  LLM answer
```

**Task (flagship):** Which MOFA factor is most associated with breast-cancer
subtype, and what drives it?

### Model backend — bring-your-own-key

Reads `ANTHROPIC_API_KEY` from `.env`; run with the **`eccb`** kernel. Only the
agent-run cell in Section 6 spends API tokens — every cell above it is local
(the server subprocess loads the cached model; no LLM involved).

### Why MCP here?

For this practical the MCP server buys little over the local functions in
notebook 01 — and that's the point: the *task* is identical, so the only thing
that changes is the **interface**. MCP starts to pay off when the tools front
something you don't want inside every notebook process — here, a **1.1 GB**
`omics.pkl` and a fitted model. The server loads that **once** at startup and
exposes only small, audited, read-only operations; many clients can reuse it.

## Learning objectives

By the end of this notebook you should be able to:
- TODO

## 0. Environment & API key

In [1]:
import asyncio
import threading

def run_with_proactor(coro):
    """Run an async coroutine on a fresh ProactorEventLoop in a separate thread.
    Needed on Windows because ipykernel's main loop is a SelectorEventLoop
    (required by tornado), which cannot spawn subprocesses.
    """
    result_box = {}

    def runner():
        loop = asyncio.WindowsProactorEventLoopPolicy().new_event_loop()
        try:
            asyncio.set_event_loop(loop)
            result_box["result"] = loop.run_until_complete(coro)
        except Exception as e:
            result_box["error"] = e
        finally:
            loop.close()

    t = threading.Thread(target=runner)
    t.start()
    t.join()

    if "error" in result_box:
        raise result_box["error"]
    return result_box["result"]

In [2]:
from pathlib import Path
import os, sys, json

from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env")

if not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY not found. Add it to a .env in the project root.")

print("OPENAI_API_KEY loaded:", bool(os.environ.get("OPENAI_API_KEY")))

SERVER_PATH = PROJECT_ROOT / "server" / "mofa_mcp_server.py"
print("MCP server:", SERVER_PATH.name)

OPENAI_API_KEY loaded: True
MCP server: mofa_mcp_server.py


## 1. The MCP server

`server/mofa_mcp_server.py` uses **FastMCP** to expose three MCP primitives, all
backed by the cached MOFA model:

- **Tools** (model-callable, read-only): `data_summary`, `split_summary`,
  `active_factors`, `factor_view_r2`, `factor_subtype_association`,
  `top_features_for_factor`, `classify_subtype_from_factors`,
  `train_vs_test_subtype_association`
- **Resource**: `mofa://summary` — a compact model/cohort summary to read into context
- **Prompt**: `interpret_factor(factor)` — a reusable interpretation template

The server loads the aligned omics + fitted model **once at startup**; `fit_mofa`
is not exposed. The clients below launch it as a stdio subprocess.

## 2. Under the hood: talk to MCP directly (no adapter)

Before the LangChain adapter, connect with the MCP SDK's own client to see what
MCP *is*: open a `ClientSession` over stdio, handshake, enumerate every
primitive, then call a tool over the wire. (No LLM — this is free.)

In [3]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
import os

server_params = StdioServerParameters(command=sys.executable, args=[str(SERVER_PATH)])

with open(os.devnull, "w") as devnull:  # I had to add this for Windows
    async with stdio_client(server_params, errlog=devnull) as (read, write):
        async with ClientSession(read, write) as session:
            init = await session.initialize()
            print(f"Connected to: {init.serverInfo.name}")
            caps = init.capabilities
            print("Server advertises:",
                "tools" if caps.tools else "", "resources" if caps.resources else "",
                "prompts" if caps.prompts else "")

            print("\n--- TOOLS (the model may choose to call these) ---")
            for t in (await session.list_tools()).tools:
                args = ", ".join(t.inputSchema.get("properties", {}))
                ro = getattr(t.annotations, "readOnlyHint", None)
                print(f"  {t.name}({args})   readOnlyHint={ro}")

            print("\n--- RESOURCES (data to read into context) ---")
            for r in (await session.list_resources()).resources:
                print(f"  {r.uri}  ({r.name})")
            summary = await session.read_resource("mofa://summary")
            print("  read mofa://summary ->", summary.contents[0].text[:80].replace(chr(10), " "), "...")

            print("\n--- PROMPTS (server-provided templates) ---")
            for p in (await session.list_prompts()).prompts:
                args = ", ".join(a.name for a in (p.arguments or []))
                print(f"  {p.name}({args})")

            print("\n--- tools/call factor_subtype_association ---")
            called = await session.call_tool("factor_subtype_association", {})
            print("  ", called.content[0].text[:120].replace(chr(10), " "))

Connected to: eccb2026-mofa
Server advertises: tools resources prompts

--- TOOLS (the model may choose to call these) ---
  data_summary()   readOnlyHint=True
  split_summary()   readOnlyHint=True
  active_factors()   readOnlyHint=True
  factor_view_r2(factor)   readOnlyHint=True
  factor_subtype_association()   readOnlyHint=True
  top_features_for_factor(factor, view, n)   readOnlyHint=True
  classify_subtype_from_factors()   readOnlyHint=True
  train_vs_test_subtype_association()   readOnlyHint=True

--- RESOURCES (data to read into context) ---
  mofa://summary  (mofa_summary)
  read mofa://summary -> {   "n_patients": 603,   "views": [     "transcriptomics",     "proteomics",     ...

--- PROMPTS (server-provided templates) ---
  interpret_factor(factor)

--- tools/call factor_subtype_association ---
   {   "factor": "Factor2",   "eta_squared": 0.743 }


The block above never imported LangChain — that is MCP directly:

- **JSON-RPC 2.0 over a transport.** Here the transport is `stdio`; it could be
  HTTP. On that wire MCP defines methods — `initialize`, `tools/list`,
  `tools/call`, `resources/list`, `resources/read`, `prompts/list`,
  `prompts/get`. Layering: **stdio/HTTP → JSON-RPC → MCP semantics**.
- **Tools are one primitive of several:**

| Primitive | Driven by | What it is | In our server |
|-----------|-----------|------------|---------------|
| **Tools** | the model | functions the model may *choose* to call | the 8 analysis functions |
| **Resources** | the app/user | data to *read into context* | `mofa://summary` |
| **Prompts** | the user | reusable, parameterized templates | `interpret_factor` |
| **Sampling** | the server | server asks the *client's* LLM to generate | (not used) |
| **Elicitation** | the server | server asks the *human* mid-call | (not used) |
| **Roots** | the client | filesystem/scope boundaries | (not used) |

- **LLM-facing metadata:** each tool carried `readOnlyHint=True`. A client can
  use that to auto-run vs. pause for human approval — meaningful precisely
  because a non-deterministic model is the caller. Generic RPC has no such notion.

## 3. Framework adaptation: `langchain-mcp-adapters`

We call **`get_tools()`**, so only the *tools* primitive flows into the agent.
The same client also exposes `get_resources()` / `get_prompt()`. Nothing about
MCP is thrown away — the adapter **is** an MCP client speaking the identical wire
protocol as section 2; `get_tools()` just converts the one primitive a
tool-calling loop consumes.

In [4]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
import os

server_params = StdioServerParameters(command=sys.executable, args=[str(SERVER_PATH)])

async def _explore_mcp():
    with open(os.devnull, "w") as devnull:  # I had to add this for Windows
        async with stdio_client(server_params, errlog=devnull) as (read, write):
            async with ClientSession(read, write) as session:
                init = await session.initialize()
                print(f"Connected to: {init.serverInfo.name}")
                caps = init.capabilities
                print("Server advertises:",
                    "tools" if caps.tools else "", "resources" if caps.resources else "",
                    "prompts" if caps.prompts else "")

                print("\n--- TOOLS (the model may choose to call these) ---")
                for t in (await session.list_tools()).tools:
                    args = ", ".join(t.inputSchema.get("properties", {}))
                    ro = getattr(t.annotations, "readOnlyHint", None)
                    print(f"  {t.name}({args})   readOnlyHint={ro}")

                print("\n--- RESOURCES (data to read into context) ---")
                for r in (await session.list_resources()).resources:
                    print(f"  {r.uri}  ({r.name})")
                summary = await session.read_resource("mofa://summary")
                print("  read mofa://summary ->", summary.contents[0].text[:80].replace(chr(10), " "), "...")

                print("\n--- PROMPTS (server-provided templates) ---")
                for p in (await session.list_prompts()).prompts:
                    args = ", ".join(a.name for a in (p.arguments or []))
                    print(f"  {p.name}({args})")

                print("\n--- tools/call factor_subtype_association ---")
                called = await session.call_tool("factor_subtype_association", {})
                print("  ", called.content[0].text[:120].replace(chr(10), " "))

run_with_proactor(_explore_mcp())

Connected to: eccb2026-mofa
Server advertises: tools resources prompts

--- TOOLS (the model may choose to call these) ---
  data_summary()   readOnlyHint=True
  split_summary()   readOnlyHint=True
  active_factors()   readOnlyHint=True
  factor_view_r2(factor)   readOnlyHint=True
  factor_subtype_association()   readOnlyHint=True
  top_features_for_factor(factor, view, n)   readOnlyHint=True
  classify_subtype_from_factors()   readOnlyHint=True
  train_vs_test_subtype_association()   readOnlyHint=True

--- RESOURCES (data to read into context) ---
  mofa://summary  (mofa_summary)
  read mofa://summary -> {   "n_patients": 603,   "views": [     "transcriptomics",     "proteomics",     ...

--- PROMPTS (server-provided templates) ---
  interpret_factor(factor)

--- tools/call factor_subtype_association ---
   {   "factor": "Factor2",   "eta_squared": 0.743 }


In [5]:
async def get_resource(uri):
    with open(os.devnull, "w") as devnull:
        async with stdio_client(
            server_params,
            errlog=devnull
        ) as (read, write):

            async with ClientSession(read, write) as session:
                await session.initialize()

                return await session.read_resource(uri)

In [6]:
res = run_with_proactor(
    get_resource("mofa://summary")
)

print(res.contents[0].text)

{
  "n_patients": 603,
  "views": [
    "transcriptomics",
    "proteomics",
    "methylation"
  ],
  "n_factors": 10,
  "active_factors": [
    "Factor1",
    "Factor2",
    "Factor3",
    "Factor4",
    "Factor5",
    "Factor6",
    "Factor7",
    "Factor8",
    "Factor9",
    "Factor10"
  ],
  "subtypes": {
    "LumA": 322,
    "LumB": 118,
    "Basal": 97,
    "Her2": 41,
    "Normal": 25
  }
}


## 4. Sanity-check one tool (no LLM)

Call a tool straight through the adapter so the structured evidence is visible
before involving the model.

In [ ]:
def parse_mcp(result):
    '''
    Write the function that from MCP tools return text content blocks; json.loads each back to a dict/list. Use the provided structure.
    
    '''
    if isinstance(result, list):
        '''

        #################################################
                        YOUR CODE HERE
        #################################################
            
        '''
    return _unwrap(parsed)

def _unwrap(parsed):
    """Undo the {"result": [...]} auto-wrapping MCP applies to non-object tool outputs."""
    if isinstance(parsed, dict) and list(parsed.keys()) == ["result"]:
        return parsed["result"]
    return parsed

class MCPTool:
    def __init__(self, name):
        self.name = name

    async def ainvoke(self, arguments):
        with open(os.devnull, "w") as devnull:
            async with stdio_client(server_params, errlog=devnull) as (read, write):
                async with ClientSession(read, write) as session:
                    await session.initialize()
                    result = await session.call_tool(self.name, arguments)
                    parsed = [json.loads(block.text) for block in result.content]
                    # if the tool returns a single object rather than a list of records,
                    # collapse back down to that one object instead of a 1-item list
                    if len(parsed) == 1:
                        return parsed[0]
                    return parsed

In [8]:
tools_by_name = {
    "factor_subtype_association": MCPTool("factor_subtype_association"),
    "classify_subtype_from_factors": MCPTool("classify_subtype_from_factors"),
}

In [9]:
assoc = await tools_by_name["factor_subtype_association"].ainvoke({})
print("factor_subtype_association (top 3):", assoc[:3])

cls = await tools_by_name["classify_subtype_from_factors"].ainvoke({})
print("classify_subtype_from_factors    :", cls)

factor_subtype_association (top 3): [{'factor': 'Factor2', 'eta_squared': 0.743}, {'factor': 'Factor1', 'eta_squared': 0.35}, {'factor': 'Factor7', 'eta_squared': 0.264}]
classify_subtype_from_factors    : {'metrics': {'accuracy': 0.828, 'balanced_accuracy': 0.741, 'macro_f1': 0.747}, 'most_confused': {'true': 'LumA', 'predicted': 'LumB', 'count': 9}}


## 5. Bind the MCP tools to Claude

Identical to notebook 01 — `bind_tools` doesn't care the tools came from an MCP
server. Binding does not call the API.

In [10]:
from langchain_core.tools import StructuredTool
from pydantic import create_model

async def _list_mcp_tools():
    with open(os.devnull, "w") as devnull:
        async with stdio_client(server_params, errlog=devnull) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                return (await session.list_tools()).tools

mcp_tool_specs = run_with_proactor(_list_mcp_tools())

def _make_langchain_tool(spec, mcp_tool: MCPTool):
    async def _run(**kwargs):
        return await mcp_tool.ainvoke(kwargs)
    return StructuredTool.from_function(
        coroutine=_run,
        name=spec.name,
        description=spec.description or spec.name,
        args_schema=None,  # LangChain will infer a permissive schema; fine for demo use
    )

tools = [
    _make_langchain_tool(spec, MCPTool(spec.name))
    for spec in mcp_tool_specs
]
tools_by_name = {t.name: t for t in tools}

In [11]:
from langchain_openai import ChatOpenAI

MODEL = "gpt-4.1"
llm = ChatOpenAI(model=MODEL, temperature=0)
llm_with_tools = llm.bind_tools(tools)
print("Bound", len(tools), "MCP tools to", MODEL)

Bound 8 MCP tools to gpt-4.1


## 6. Run the agent loop (async)  ⟵ *these cells spend API tokens*

Same loop as notebook 01, but `await`-ing the model and tools because MCP tools
are asynchronous.

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage

SYSTEM = ("You are a computational-biology assistant analysing a fitted MOFA model of "
          "TCGA breast-cancer multi-omics data. Use the tools to gather evidence; ground "
          "every quantitative claim in tool results and name which tool you used. Factors "
          "are 'Factor1'..'Factor10'; subtypes are PAM50 (LumA, LumB, Basal, Her2, Normal).")

async def run_agent(question: str, max_steps: int = 6, verbose: bool = True) -> AIMessage:
    '''
    #################################################
                    YOUR CODE HERE
    #################################################

    Write the function to run the agent. Additional clue: Use the structure and previous notebook as reference/inspiration.
    '''

    messages = '''YOUR CODE HERE'''
    for step in range(max_steps):
        '''
        #################################################
                        YOUR CODE HERE
        #################################################
        '''
        
    raise RuntimeError(f"Agent did not finish within {max_steps} steps.")

In [13]:
from typing import Any
from pydantic import create_model

def _json_schema_to_pydantic(name: str, input_schema: dict):
    """Turn an MCP tool's JSON schema into a Pydantic model LangChain can use."""
    props = input_schema.get("properties", {})
    required = set(input_schema.get("required", []))
    type_map = {"string": str, "integer": int, "number": float, "boolean": bool}

    fields = {}
    for prop_name, prop_spec in props.items():
        py_type = type_map.get(prop_spec.get("type"), Any)
        default = ... if prop_name in required else prop_spec.get("default", None)
        fields[prop_name] = (py_type, default)

    return create_model(f"{name}_Args", **fields)


def _make_langchain_tool(spec, mcp_tool: MCPTool):
    ArgsModel = _json_schema_to_pydantic(spec.name, spec.inputSchema)

    async def _run(**kwargs):
        return await mcp_tool.ainvoke(kwargs)

    return StructuredTool.from_function(
        coroutine=_run,
        name=spec.name,
        description=spec.description or spec.name,
        args_schema=ArgsModel,
    )

In [14]:
tools = [_make_langchain_tool(spec, MCPTool(spec.name)) for spec in mcp_tool_specs]
tools_by_name = {t.name: t for t in tools}
llm_with_tools = llm.bind_tools(tools)

In [ ]:
final = await run_agent('''

        #################################################
                        YOUR CODE HERE
        #################################################
        
        Ask the agent about specific MOFA factors associated with breast-cancer subtypes, that are stringly driven by transcription factors.
        '''
   )
print("\n=== FINAL ANSWER ===\n")
print(final.content)

[step 0] -> factor_subtype_association({})
[step 0] -> top_features_for_factor({'factor': 'Factor2', 'view': 'transcriptomics', 'n': 5})

=== FINAL ANSWER ===

Based on the results from the factor_subtype_association tool, Factor 2 is the MOFA factor most strongly associated with breast-cancer subtype (PAM50), with an eta-squared value of 0.743.

The top transcriptomic features driving Factor 2, according to the top_features_for_factor tool, are:

Top positive drivers:
- ENSG00000160182.3 (weight: 0.470)
- ENSG00000173467.9 (weight: 0.469)
- ENSG00000082175.15 (weight: 0.436)
- ENSG00000235687.9 (weight: 0.424)
- ENSG00000160180.15 (weight: 0.417)

Top negative drivers:
- ENSG00000166535.20 (weight: -0.337)
- ENSG00000186832.9 (weight: -0.301)
- ENSG00000102243.13 (weight: -0.283)
- ENSG00000185686.18 (weight: -0.279)
- ENSG00000135069.14 (weight: -0.278)

All quantitative claims are grounded in the results of the factor_subtype_association and top_features_for_factor tools.


## 7. Same answer as the direct-tools track

The flagship answer should match [`01_tools_v1`](01_tools_v1.ipynb) — same
backend and cached model, different transport. Compare the factor the agent
picked against the direct evidence:

In [ ]:
print("Direct evidence (top subtype-associated factors):")
for row in assoc[:3]:
    print(f"  {row['factor']}: eta_squared = {row['eta_squared']}")

Direct evidence (top subtype-associated factors):
  Factor2: eta_squared = 0.743
  Factor1: eta_squared = 0.35
  Factor7: eta_squared = 0.264


Direct evidence (top subtype-associated factors):
  Factor2: eta_squared = 0.743
  Factor1: eta_squared = 0.35
  Factor7: eta_squared = 0.264


## Reflection

- **Primitives:** name the three this server exposes. Which adapter calls load
  the resource and the prompt instead of the tools? (`get_resources()`, `get_prompt()`)
- **Did the adapter throw MCP away?** No — it *is* an MCP client speaking the
  section-2 protocol. What actually differs between sections 2 and 3? (convenience
  and object model, not capability.)
- **Layering:** MCP rode on `stdio` + JSON-RPC. What changes if the transport is
  HTTP? (the primitives don't.)
- **Annotations:** every tool was `readOnlyHint=True`. How would a client use
  that? What changes for a tool that *writes*?
- **Same answer:** it matched notebook 01. So what did MCP buy here, and when does
  it pay off? (big/private data behind an audited server, reuse across clients.)

## 8. Bringing in an external MCP server — BioMCP

So far our only MCP server has been our *own* — a private, local server fronting
our own fitted model and data. That's one end of the spectrum. The other end:
**public, third-party MCP servers that expose general biomedical knowledge** —
we don't own or run them, we just connect to them as a client, the same way we
connect to `mofa_mcp_server.py`.

[**BioMCP**](https://biomcp.org) is a real example: an open-source MCP server
that fronts ~15 public biomedical data sources (PubMed, ClinicalTrials.gov,
MyGene.info, MyChem.info, MyDisease.info, ClinVar, and others) behind a
consistent set of tools — gene/drug/disease lookups, literature search, trial
search, and more.

**Why this matters pedagogically:** it's the same MCP client code, connecting
to a completely different kind of server — one we didn't write, running
somewhere else, fronting public knowledge instead of our private model. This is
exactly the "dataset lookup" / external-server angle from the session plan:
the client only needs an MCP client, regardless of who runs the server or
what's behind it.

### Install BioMCP

```bash
pip install biomcp-python
# or, for the newer single-binary CLI: uv tool install biomcp-cli
```

### Two servers, one agent

We now give the agent **two** MCP servers at once: our private `mofa` server
(local data) and the public `biomcp` server (general biomedical knowledge).
`MultiServerMCPClient` supports this natively — just add a second entry.


In [ ]:
import shutil, subprocess, sys
from langchain_mcp_adapters.client import MultiServerMCPClient

BIOMCP_BIN = shutil.which("biomcp")
assert BIOMCP_BIN, "biomcp CLI not found on PATH — install it first: pip install biomcp-cli (or biomcp-python), then restart the kernel"
print("biomcp binary:", BIOMCP_BIN)

help_text = subprocess.run([BIOMCP_BIN, "--help"], capture_output=True, text=True).stdout
biomcp_subcommand = '''YOUR CODE HERE ### Write the code to configure the biomcop command'''
print("Using subcommand:", biomcp_subcommand)

multi_server_client = MultiServerMCPClient({
    "mofa": {"command": sys.executable, "args": [str(SERVER_PATH)], "transport": "stdio"},
    "biomcp": {"command": BIOMCP_BIN, "args": [biomcp_subcommand], "transport": "stdio"},
})

biomcp binary: c:\Users\elisa\Documents\mcp\ECCB2026_TEST\.venv\Scripts\biomcp.EXE
Using subcommand: serve


### Re-bind and re-define the agent loop over the combined tool set

Nothing about `bind_tools` or `run_agent` changes conceptually — the model just
now has a larger, mixed set of tools to choose from, some fronting our own
model, some fronting public biomedical databases.


In [ ]:
async def _get_all_tools():
    return await multi_server_client.get_tools()

all_tools = '''YOUR CODE HERE ### Use existing function to assign this variable'''
all_tools_by_name = {t.name: t for t in all_tools}

mofa_tool_names = set(tools_by_name)          # from Section 3 (mofa-only client)
biomcp_tool_names = set(all_tools_by_name) - mofa_tool_names

print(f"Loaded {len(all_tools)} tools total")
print("  from mofa   :", sorted(mofa_tool_names))
print("  from biomcp :", sorted(biomcp_tool_names))

Loaded 11 tools total
  from mofa   : ['active_factors', 'classify_subtype_from_factors', 'data_summary', 'factor_subtype_association', 'factor_view_r2', 'split_summary', 'top_features_for_factor', 'train_vs_test_subtype_association']
  from biomcp : ['biomcp', 'get', 'search']


In [42]:
llm_with_all_tools = llm.bind_tools(all_tools)
print("Bound", len(all_tools), "tools (mofa + biomcp) to", MODEL)

SYSTEM_MULTI = (
    "You are a computational-biology assistant. You have two kinds of tools: "
    "(1) MOFA tools analysing a fitted multi-omics model of TCGA breast-cancer "
    "data (factors 'Factor1'..'Factor10', PAM50 subtypes), and (2) BioMCP tools "
    "for general biomedical knowledge (genes, drugs, diseases, literature). "
    "Use MOFA tools for questions about our fitted model/factors/patients; use "
    "BioMCP tools for general biomedical facts about specific genes, drugs, or "
    "diseases. Ground every claim in tool output and name which tool you used. "
    "NOTE: MOFA tools return Ensembl gene IDs with a version suffix (e.g. "
    "'ENSG00000160180.15'). BioMCP tools do not recognise the version suffix — "
    "strip it (to 'ENSG00000160180') before passing an ID to any BioMCP tool. "
    "Resolve one gene at a time rather than issuing many lookups in parallel." \
    "If a question asks for something no available tool result actually supports, "
    "say so explicitly rather than inferring an answer from general biomedical knowledge."
)

async def run_agent_multi(question: str, max_steps: int = 6, verbose: bool = True) -> AIMessage:
    messages = [SystemMessage(content=SYSTEM_MULTI), HumanMessage(content=question)]
    for step in range(max_steps):
        ai = await llm_with_all_tools.ainvoke(messages)
        messages.append(ai)
        if not ai.tool_calls:
            return ai
        for call in ai.tool_calls:
            if verbose:
                print(f"[step {step}] -> {call['name']}({call['args']})")
            result = await all_tools_by_name[call["name"]].ainvoke(call["args"])
            if verbose:
                # Spot-check what actually came back over the wire — this is the
                # evidence for the privacy claim in the closing markdown: only the
                # arguments above went OUT to BioMCP; this is what came BACK.
                preview = json.dumps(result, default=str)
                if len(preview) > 300:
                    preview = preview[:300] + f"... [{len(preview)} chars total]"
                print(f"           <- {preview}")
            messages.append(ToolMessage(content=json.dumps(result, default=str),
                                        tool_call_id=call["id"]))
    raise RuntimeError(f"Agent did not finish within {max_steps} steps.")

Bound 11 tools (mofa + biomcp) to gpt-4.1


In [40]:
final_multi = run_with_proactor(run_agent_multi(
    """Factor2 is associated with breast cancer subtype. What biological processes are enriched among its top 50 genes?""",
    max_steps=12,
))


print("\n=== FINAL ANSWER ===\n")
print(final_multi.content)

[step 0] -> top_features_for_factor({'factor': 'Factor2', 'n': 50})
           <- [{"type": "text", "text": "{\n  \"factor\": \"Factor2\",\n  \"view\": \"transcriptomics\",\n  \"top_negative\": {\n    \"ENSG00000166535.20\": -0.337,\n    \"ENSG00000186832.9\": -0.301,\n    \"ENSG00000102243.13\": -0.283,\n    \"ENSG00000185686.18\": -0.279,\n    \"ENSG00000135069.14\": -0.278,\n ... [3800 chars total]
[step 1] -> biomcp({'command': 'enrich ENSG00000166535,ENSG00000186832,ENSG00000102243,ENSG00000185686,ENSG00000135069,ENSG00000102854,ENSG00000105141,ENSG00000164434,ENSG00000143546,ENSG00000136928,ENSG00000205420,ENSG00000156219,ENSG00000107159,ENSG00000176887,ENSG00000198729,ENSG00000280916,ENSG00000163362,ENSG00000163220,ENSG00000173894,ENSG00000026559,ENSG00000189001,ENSG00000196611,ENSG00000229544,ENSG00000143452,ENSG00000163064,ENSG00000117148,ENSG00000135374,ENSG00000167656,ENSG00000105173,ENSG00000114805,ENSG00000101057,ENSG00000188910,ENSG00000144354,ENSG00000166426,ENSG00000119

In [44]:
final_multi = run_with_proactor(run_agent_multi(
    """Run BioMCP's gene-set enrichment on Factor2's top 5 genes and report only the pathways/processes it returns.""",
    max_steps=12,
))


print("\n=== FINAL ANSWER ===\n")
print(final_multi.content)

[step 0] -> top_features_for_factor({'factor': 'Factor2', 'n': 5})
           <- [{"type": "text", "text": "{\n  \"factor\": \"Factor2\",\n  \"view\": \"transcriptomics\",\n  \"top_negative\": {\n    \"ENSG00000166535.20\": -0.337,\n    \"ENSG00000186832.9\": -0.301,\n    \"ENSG00000102243.13\": -0.283,\n    \"ENSG00000185686.18\": -0.279,\n    \"ENSG00000135069.14\": -0.278\n  ... [560 chars total]
[step 1] -> biomcp({'command': 'enrich ENSG00000160180,ENSG00000235687,ENSG00000082175,ENSG00000173467,ENSG00000160182', 'json': True})
           <- [{"type": "text", "text": "{\n  \"genes\": [\n    \"ENSG00000160180\",\n    \"ENSG00000235687\",\n    \"ENSG00000082175\",\n    \"ENSG00000173467\",\n    \"ENSG00000160182\"\n  ],\n  \"count\": 10,\n  \"results\": [\n    {\n      \"native\": \"WP:WP5390\",\n      \"name\": \"Pancreatic cancer subtyp... [2053 chars total]

=== FINAL ANSWER ===

BioMCP gene-set enrichment for Factor2's top 5 genes returns the following pathways/processes:

1. Pa

### Possible Problems to Discuss

#### Is it safe to run the external server on our dataset?

- **The raw multi-omics data and the fitted MOFA model never go to BioMCP.**
  They live only inside `mofa_mcp_server.py`'s process, loaded once from local
  cache. BioMCP has no access to that server or its data — it's a completely
  separate subprocess with no shared state.
- **What actually reaches BioMCP is whatever the LLM puts into a tool call's
  arguments** — in the trace above, that's Ensembl gene IDs
  (`ENSG00000160180`) or gene symbols. BioMCP then relays those terms out to
  public APIs (PubMed, ClinicalTrials.gov, MyGene.info, g:Profiler, etc.), so
  those third-party services see the query terms too.
- Separately — and this is true regardless of BioMCP — **the full
  conversation, including MOFA tool results, already goes to the model
  provider's API** as part of the agent loop. That's a standard consideration
  for any LLM tool-calling setup, not specific to adding BioMCP.

**Why this is low-risk *for this dataset specifically*:** TCGA breast-cancer
data is a public, de-identified research cohort — gene names, factor
loadings, and PAM50 subtype labels derived from it aren't sensitive on their
own. So in this notebook, the practical exposure is small.

Two things worth knowing if you're adapting this to a less public dataset:

1. **The system prompt is a guideline, not a technical boundary.** Nothing
   stops the model from putting more context into a `search` or `biomcp`
   call than just a gene ID — e.g. it could phrase a query mentioning cohort
   size or subtype distribution if it thought that was helpful. The prompt
   discourages this but doesn't enforce it.
2. **You can enforce it in code** if you want a hard guarantee: wrap the
   BioMCP tool calls with a validator that only allows arguments matching an
   expected pattern (bare Ensembl IDs, HGNC symbols, drug/disease names) and
   rejects anything else before it reaches `ainvoke`.

*(Live demo tip: the printed tool results in the cell above are the evidence
for this — you can point at exactly what went out and what came back, rather
than asking the audience to take the claim on faith.)*

#### A separate, unrelated problem: Windows event loops

This one bit us earlier in the notebook and is worth flagging on its own,
since it's an engineering gotcha rather than a safety one. MCP's
`stdio_client` spawns the server as a subprocess using `anyio`, which on
Windows requires an event loop that supports subprocess creation
(`ProactorEventLoop`). Jupyter kernels on Windows default to a
`SelectorEventLoop` instead (needed for the kernel's own messaging), which
cannot spawn subprocesses at all. That mismatch — not anything about MCP
itself — is the root cause of `run_with_proactor` existing in cell 3.
